# 第 9 章 基于宇树开源的 Qmini 双足机器人


## 9.4.1 软件栈与 URDF 模型


Qmini 的软件系统以 URDF 模型为机器人描述的基础，并围绕核心软件栈 RoboTamer4Qmini 展开。URDF 使用 XML 格式描述机器人的连杆（link）、关节（joint）以及各部件之间的运动学关系与质量 / 惯量属性。一个简化的 URDF 关节描述如下：



In [ ]:
<link name="hip_link">
    <inertial>
        <mass value="0.5"/>
        <inertia ixx="0.001" ixy="0" ixz="0"
                 iyy="0.001" iyz="0"
                 izz="0.001"/>
    </inertial>
</link>

<joint name="hip_joint" type="revolute">
    <parent link="base_link"/>
    <child link="hip_link"/>
    <origin xyz="0 0 0.15" rpy="0 0 0"/>
    <axis xyz="0 1 0"/>
    <limit lower="-1.57" upper="1.57" effort="10" velocity="3"/>
</joint>


双足行走的核心难题在于**动态平衡**。与四足或多足机器人不同，双足机器人在行走的大部分时间里处于静态或动态失稳状态，需要控制器持续调节身体姿态与落脚点，才能维持稳定。

在双足行走控制中，一个经典的分析工具是**零力矩点（Zero Moment Point, ZMP）**。当机器人处于单腿支撑阶段时，只要 ZMP 落在支撑脚与地面构成的支撑多边形（Support Polygon）内，机器人就能保持稳定。在简化的线性倒立摆模型（LIPM）下，质心在水平方向的位置 $x$ 与 ZMP 位置 $p_x$ 满足

$p_x = x - \frac{z_c}{g}\,\ddot{x} \tag{9.1}$

其中，$z_c$ 为质心的高度，$g$ 为重力加速度，$\ddot{x}$ 为质心的水平加速度。式（9.1）表明，通过合理规划质心的运动轨迹，可以控制 ZMP 始终处于支撑多边形内，从而保证行走的稳定性。

在工程实现上，Qmini 的步态控制采用 \*\* 深度强化学习（Deep Reinforcement Learning, DRL）\*\* 方法：其运动控制解决方案基于 NVIDIA 的 Isaac Gym 仿真环境，利用深度强化学习让机器人学习稳健的步态技能，包括在不平坦地形上行走。为了平滑实现从仿真到现实的迁移（Sim-to-Real），训练过程中采用了随机化动力学参数、随机化环境扰动等关键技术，使在仿真中训练出的策略能够直接部署到真实机器人上。

从强化学习的角度，策略训练的目标是最大化累计回报

$J(\theta) = \mathbb{E}_{\tau \sim \pi_{\theta}} \left[ \sum_{t=0}^{T} \gamma^{t} r(s_{t}, a_{t}) \right] \tag{9.2}$

其中，$s_{t}$ 为第 $t$ 步的机器人状态（关节角度、角速度、姿态等），$a_{t}$ 为策略 $\pi_{\theta}$ 输出的关节力矩指令，$r(s_{t}, a_{t})$ 为奖励函数（例如鼓励前进速度、抑制躯干倾斜与能耗），$\gamma$ 为折扣因子。通过在 Isaac Gym 中并行仿真大量机器人，深度强化学习算法能够高效地探索出稳健的双足步态策略。



## 9.4.4 基于 unitree\_sdk2 的二次开发


宇树提供了 **unitree\_sdk2**（含 Python 接口 unitree\_sdk2\_python）作为机器人的二次开发工具。unitree\_sdk2 是对 DDS（Data Distribution Service）的一层封装，通过发布 / 订阅（Publish/Subscribe）和请求 / 响应（Request/Response）的方式，实现上位机与机器人内部 / 外部进程之间的数据通信。

下面给出一个基于 unitree\_sdk2 Python 接口、通过底层数据通道读取并控制关节的示例程序。注意，Qmini 的具体主题（Topic）名称与通道配置请以宇树官方 Qmini 开发者文档为准，以下代码展示的是 unitree\_sdk2 人形 / 双足机器人底层接口的通用模式。



In [ ]:
import numpy as np
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_, LowState_

# 初始化 DDS 通道（0 为默认域，eth0 为网卡名）
ChannelFactoryInitialize(0, "eth0")

# 定义低层状态回调：读取机器人各关节角度与角速度
def low_state_handler(msg):
    for i in range(msg.motor_state_size):   # 遍历各关节
        q  = msg.motor_state[i].q            # 关节角度
        dq = msg.motor_state[i].dq           # 关节角速度
        tau_est = msg.motor_state[i].tau_est # 估计力矩
        print(f"joint {i}: q={q:.3f} dq={dq:.3f}")

# 订阅低层状态主题
state_sub = ChannelSubscriber("rt/lowstate", LowState_)
state_sub.Init(low_state_handler, 10)

# 发布低层控制指令主题
cmd_pub = ChannelPublisher("rt/lowcmd", LowCmd_)
cmd_pub.Init()

cmd = LowCmd_()
cmd.motor_cmd_size = cmd.motor_state_size  # 与电机数量一致

# 循环发送关节力矩/位置指令
for step in range(1000):
    for i in range(cmd.motor_cmd_size):
        cmd.motor_cmd[i].tau    = 0.0                 # 期望力矩
        cmd.motor_cmd[i].q      = np.sin(step * 0.01) # 期望关节位置
        cmd.motor_cmd[i].dq     = 0.0                 # 期望关节角速度
        cmd.motor_cmd[i].kp     = 20.0                # 位置增益
        cmd.motor_cmd[i].kd     = 1.0                 # 速度增益
    cmd_pub.Write(cmd)


本章介绍了宇树科技开源的低成本双足机器人 Qmini。Qmini 以完全开源、低成本、模块化为特色，通过 3D 打印与标准件采购即可复刻，总成本可控制在万元以内。本章详细阐述了 Qmini 的软硬件架构（树莓派主控、8010 电机、485 总线电气连接）、四步制作流程、基于 URDF 与 ROS/Gazebo 的建模与仿真方法，以及基于零力矩点和深度强化学习的双足运动控制原理，并给出了基于 unitree\_sdk2 的二次开发示例。与上一章的 SO-101 机械臂相呼应，Qmini 为读者提供了从 "操作型" 到 "移动型" 的低成本机器人学习完整路径，也为后续深入研究具身智能与机器人运动控制奠定了平台基础。

